# Kasafi Bot on Google Colab
Run the whole project here (fast model downloads + a big GPU) while the laptop catches up.

**Before you start**
1. **Runtime → Change runtime type → GPU** (L4 or A100 if you have Colab Pro; T4 also works).
2. Open **Secrets** (🔑 key icon, left sidebar), add these and switch on *Notebook access* for each:
   - `HF_TOKEN`: your Hugging Face read token
   - `TELEGRAM_BOT_TOKEN`: from @BotFather (only needed for step 7)
   - `LOG_SALT`: the **same** random string as in your laptop `.env`
   - `ADMIN_TELEGRAM_IDS`: your Telegram id (send /myid to the bot) for /stats, /recent, /bad
   - `TWILIO_ACCOUNT_SID`, `TWILIO_AUTH_TOKEN`: only for WhatsApp (Twilio Console home page)
   - `GITHUB_TOKEN`: a GitHub fine-grained token with read access to `kasafi-bot` (or skip it and upload `kasafi-bot.zip` to `My Drive/kasafi/`)
3. Optional: put your test voice note at `My Drive/kasafi/test-voice.ogg`.

Models are saved to **`My Drive/kasafi/hf_cache`** (about 17 GB, so check you have that much free Drive space). Later sessions copy them from Drive instead of downloading again.

In [ ]:
# 1. Mount Drive and set paths
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, shutil, subprocess
DRIVE = '/content/drive/MyDrive/kasafi'
DRIVE_CACHE = f'{DRIVE}/hf_cache'          # permanent copy of the models
LOCAL_CACHE = '/content/hf_cache'          # fast local disk used while running
os.makedirs(DRIVE_CACHE, exist_ok=True); os.makedirs(f'{DRIVE}/logs', exist_ok=True); os.makedirs(f'{DRIVE}/evidence', exist_ok=True)
os.environ['HF_HOME'] = LOCAL_CACHE
def secret(k):
    try: return userdata.get(k)
    except Exception: return None
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free space reported: {free/1e9:.0f} GB (need ~17 GB for the models)')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code (GitHub if GITHUB_TOKEN is set, otherwise My Drive/kasafi/kasafi-bot.zip)
REPO = '/content/kasafi-bot'
tok = secret('GITHUB_TOKEN')
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} pull --ff-only
elif tok:
    !git clone -q https://{tok}@github.com/bawanallahbilyaminu/kasafi-bot.git {REPO}
else:
    !unzip -oq {DRIVE}/kasafi-bot.zip -d /content/
%cd {REPO}
!git log --oneline -3 2>/dev/null || ls

In [ ]:
# 3. Install (Colab already has CUDA PyTorch)
!apt-get -qq install -y ffmpeg > /dev/null
!grep -v -i '^torch' requirements.txt > /tmp/req.txt && pip -q install -r /tmp/req.txt
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# 4. Models: copy from Drive if already saved there, otherwise download (fast) and save to Drive
from huggingface_hub import snapshot_download
import subprocess
MODELS = ['NCAIR1/Hausa-ASR', 'NCAIR1/N-ATLaS', 'facebook/mms-tts-hau']
hub_drive, hub_local = f'{DRIVE_CACHE}/hub', f'{LOCAL_CACHE}/hub'
os.makedirs(hub_local, exist_ok=True)
os.makedirs(hub_drive, exist_ok=True)

for m in MODELS:
    folder = 'models--' + m.replace('/', '--')
    done_marker = f'{hub_drive}/{folder}/.complete'
    if os.path.exists(done_marker) and not os.path.isdir(f'{hub_local}/{folder}'):
        print(f'Copying {m} from Drive...')
        subprocess.run(['cp', '-r', f'{hub_drive}/{folder}', f'{hub_local}/'], check=True)
    print(f'Checking/downloading {m}...')
    snapshot_download(m, token=secret('HF_TOKEN'))      # skips files that are already complete
    if not os.path.exists(done_marker):
        print(f'Saving {m} to Drive (a few minutes for N-ATLaS)...')
        subprocess.run(['rsync', '-a', f'{hub_local}/{folder}', f'{hub_drive}/'], check=True)
        open(done_marker, 'w').close()
    print(f'{m}: done\n')

subprocess.run(f'du -sh {hub_local}/* {hub_drive}/*', shell=True)


In [ ]:
# 5. Write .env (big GPU -> full-precision N-ATLaS, small GPU -> 4-bit like the laptop)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
env = {
  'HF_TOKEN': secret('HF_TOKEN') or '',
  'TELEGRAM_BOT_TOKEN': secret('TELEGRAM_BOT_TOKEN') or '',
  'LOG_SALT': secret('LOG_SALT') or 'set-LOG_SALT-secret',
  'HF_HOME': LOCAL_CACHE,
  'LLM_BACKEND': 'transformers',
  'LLM_4BIT': '0' if gpu_gb > 20 else '1',
  'ASR_ENABLED': '1',
  'LOG_DB': f'{DRIVE}/logs/interactions_colab.sqlite',   # evidence survives the session
  'TTS_ENABLED': '1', 'VOICE_REPLY': 'voice', 'TTS_RATE': '0.9',
  'REQUIRE_REGISTRATION': '1',
  'ADMIN_TELEGRAM_IDS': secret('ADMIN_TELEGRAM_IDS') or '',
  'TWILIO_ACCOUNT_SID': secret('TWILIO_ACCOUNT_SID') or '',
  'TWILIO_AUTH_TOKEN': secret('TWILIO_AUTH_TOKEN') or '',
  'TWILIO_WHATSAPP_FROM': 'whatsapp:+14155238886',      # Twilio sandbox number
  'ENGINE_API_KEY': secret('ENGINE_API_KEY') or '',     # for the web app (optional)
}
open('.env', 'w').write('\n'.join(f'{k}={v}' for k, v in env.items()))
os.environ.update(env)
print(f'GPU {gpu_gb:.0f} GB -> LLM_4BIT={env["LLM_4BIT"]}; log at {env["LOG_DB"]}')
!python -m tests.test_questions | tail -2
!python -I pipeline/validate.py data/db/budget.sqlite | tail -1

In [ ]:
# 6. Integration check: both official models + one Hausa round trip (saved to Drive as evidence)
audio = f'{DRIVE}/test-voice.ogg'
arg = f'--audio "{audio}"' if os.path.exists(audio) else ''
!python scripts/check_models.py {arg} 2>&1 | tee "{DRIVE}/evidence/check_models_colab.txt"

In [ ]:
# 6b. Ask questions right here (loads the models into this notebook).
# Skip this cell if you go straight to step 7, so the GPU isn't holding the models twice.
import sys; sys.path.insert(0, REPO)
from app.engine import Engine
eng = Engine()
for q in ['Nawa aka ware wa lafiya a 2026?', 'Nawa aka kashe a fannin ilimi a 2025?', 'Wadanne ayyuka ne za a yi a Tambuwal?']:
    r = eng.handle_text('colab-test', 'me', q)
    print('Q:', q, '\n' + r.text, '\n' + '-'*60)
if os.path.exists(audio):
    r = eng.handle_voice('colab-test', 'me', open(audio, 'rb').read(), '.ogg')
    print('[voice] heard:', r.transcript, '\n' + r.text)

In [ ]:
# 6c. Free the GPU before step 7 (if step 7 runs out of memory: Runtime -> Restart session, run 1, 2, 5, then 7)
import gc
try: del eng
except NameError: pass
gc.collect(); torch.cuda.empty_cache(); print('freed')

## 7. Run the bot from Colab (Telegram, plus WhatsApp if the TWILIO secrets are set)
**Stop the bot on the laptop first.** Only one copy can use a bot token at a time; otherwise Telegram returns a "Conflict" error.
Keep this tab open: the bot stops when the cell stops or Colab disconnects.

For WhatsApp, this cell opens a free Cloudflare tunnel and prints a URL. Paste it into **Twilio Console → Messaging → Try it out → Send a WhatsApp message → Sandbox settings → "When a message comes in"** (method POST), and Save. The URL changes every time this cell restarts, so paste it again after a restart.

Interactions are saved to `My Drive/kasafi/logs/interactions_colab.sqlite`. Merge them into the laptop log later with `python scripts\merge_logs.py`.


In [ ]:
import re, threading, subprocess
PORT = 8765   # not 8080: Colab's own Jupyter server uses 8080
public = ''
subprocess.run('pkill -f cloudflared', shell=True)   # close a tunnel left over from an earlier run
if env.get('TWILIO_ACCOUNT_SID') or env.get('ENGINE_API_KEY'):
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
                       '-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
    tun = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PORT}'],
                           stderr=subprocess.PIPE, text=True)
    for _ in range(200):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tun.stderr.readline())
        if m:
            public = m.group(0); break
    threading.Thread(target=lambda: [None for _ in tun.stderr], daemon=True).start()   # keep draining the log
    os.environ['PUBLIC_BASE_URL'] = public
    print('Public URL:', public)
    if env.get('TWILIO_ACCOUNT_SID'):
        print('>>> Twilio sandbox "When a message comes in":', public + '/whatsapp')
    if env.get('ENGINE_API_KEY'):
        print('>>> Web app ENGINE_URL:', public)
flag = '' if env.get('TWILIO_ACCOUNT_SID') or env.get('ENGINE_API_KEY') else '--no-whatsapp'
!python -m app.run_all {flag} --port {PORT}


## 8. Export evidence (any time, from the Drive log)

In [ ]:
!LOG_DB="{DRIVE}/logs/interactions_colab.sqlite" python scripts/export_evidence.py && cp -r evidence/* "{DRIVE}/evidence/"